# IdeoGram with Samtools and MG2C
By Zaide Montes-Ortiz

## Converting Gff3 to Map_Genes.txt


`http://mg2c.iask.in/mg2c_v2.1/`


After mapping the candidates genes to the reference genome with miniprot, we need to convert/extract the information to plot the ideogram in **MG2C**. We need to extract the following information input1: Gene location which includes five fields: 1.gene_id; 2.gene_start; 3.gene_end; 4.chr_id; 5.gene_color. for inputfile2. input2: chromosome length which includes two fields: 1.chr_id; 2.chr_length.

`Input 1 to MG2C looks like`


| gene            | gene_start | gene_end  | chr_id      | gene_color |
|-----------------|------------|-----------|-------------|------------|
| STRG.11600.1.p2 | 11753042   | 11821584  | OX387451.1  | Black      |
| STRG.11600.1.p2 | 11753042   | 11753639  | OX387451.1  | Red        |
| STRG.11600.1.p2 | 11753724   | 11754521  | OX387451.1  | Black      |
| STRG.11600.1.p2 | 11821319   | 11821584  | OX387451.1  | Red        |
| STRG.11765.2.p1 | 11742789   | 11822073  | OX387451.1  | yellow     |
| STRG.11765.2.p1 | 11820818   | 11822073  | OX387451.1  | Black      |
| STRG.11765.2.p1 | 11752398   | 11754473  | OX387451.1  | Red        |
| STRG.11765.2.p1 | 11744971   | 11745048  | OX387451.1  | Black      |
| STRG.11765.2.p1 | 11744815   | 11744943  | OX387451.1  | Red        |



In [3]:
import pandas as pd

# Function to determine the color based on type and position
def determine_color(feature_type, start, end, parent_start, parent_end):
    if feature_type == 'mRNA':
        return 'Black'
    elif feature_type == 'CDS':
        if start == parent_start or end == parent_end:
            return 'Red'
        return 'Black'
    return 'Yellow'

# Read the GFF3 file
gff3_file = '/Users/zaidemontesortiz/Documents/1_PhD_project_Spiders_Ticks_Beetles/Spiders/Analysis/IdeoZai/output_miniprot.gff'
data = []
with open(gff3_file, 'r') as file:
    for line in file:
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        seqid, source, feature_type, start, end, score, strand, phase, attributes = fields
        attr_dict = {k: v for k, v in (item.split('=') for item in attributes.split(';') if '=' in item)}
        
        if 'ID' in attr_dict:
            gene_id = attr_dict['ID']
        elif 'Parent' in attr_dict:
            gene_id = attr_dict['Parent']
        else:
            continue

        data.append({
            'gene': attr_dict.get('Target', '').split(' ')[0],
            'gene_start': int(start),
            'gene_end': int(end),
            'chr_id': seqid,
            'feature_type': feature_type
        })

# Convert to DataFrame
df = pd.DataFrame(data)

# Merge with mRNA start and end positions
mRNA_df = df[df['feature_type'] == 'mRNA'].copy()
mRNA_df = mRNA_df.rename(columns={'gene_start': 'mRNA_start', 'gene_end': 'mRNA_end'})
df = df.merge(mRNA_df[['gene', 'mRNA_start', 'mRNA_end']], on='gene', how='left')

# Apply color logic
df['gene_color'] = df.apply(lambda row: determine_color(row['feature_type'], row['gene_start'], row['gene_end'], row['mRNA_start'], row['mRNA_end']), axis=1)

# Select and reorder columns
result_df = df[['gene', 'gene_start', 'gene_end', 'chr_id', 'gene_color']]

# Drop duplicates
result_df = result_df.drop_duplicates()

# Save to CSV or print
result_df.to_csv('output_table_iGluR_IR_2025.csv', index=False)
print(result_df)




                               gene  gene_start   gene_end      chr_id  \
0    ArgIR_g10037.t1|XP_055927709.1     9444468    9647558  OX387446.1   
1    ArgIR_g10037.t1|XP_055927709.1     9647471    9647558  OX387446.1   
2    ArgIR_g10037.t1|XP_055927709.1     9626952    9627104  OX387446.1   
3    ArgIR_g10037.t1|XP_055927709.1     9587576    9587824  OX387446.1   
4    ArgIR_g10037.t1|XP_055927709.1     9570438    9570688  OX387446.1   
..                              ...         ...        ...         ...   
957     ArgIR_g8850.t1|KAF8787221.1    79110235   79110306  OX387445.1   
958                                    79110304   79110306  OX387445.1   
959     ArgIR_g9566.t1|KAF8788018.1   118335673  118336929  OX387445.1   
960     ArgIR_g9566.t1|KAF8788018.1   118335673  118336929  OX387445.1   
961                                   118335673  118335675  OX387445.1   

    gene_color  
0        Black  
1          Red  
2        Black  
3        Black  
4        Black  
..       

### To determine the start and end positions for unique gene IDs

To determine the start and end positions for unique gene IDs from your GFF3-derived table, you can aggregate the data by gene ID, and calculate the minimum start and maximum end positions for each gene. 



In [7]:

# Load the data from the CSV file
input_file = 'output_table_iGluR_IR_2025.csv'
df = pd.read_csv(input_file)

# Group by gene and calculate min start and max end for each gene
agg_df = df.groupby('gene').agg(
    gene_start=('gene_start', 'min'),
    gene_end=('gene_end', 'max'),
    chr_id=('chr_id', 'first')  # Assuming chr_id is the same for each gene
).reset_index()

# Display the aggregated DataFrame
print(agg_df)

# Save the aggregated DataFrame to a new CSV file
output_file = 'aggregated_table_iGluR_IR_2025.csv'
agg_df.to_csv(output_file, index=False)


                                                  gene  gene_start   gene_end  \
0                                      ArgIR_IRg185.t1    79095330   79096325   
1                                      ArgIR_IRg186.t1    79083874   79084749   
2                       ArgIR_g10037.t1|XP_055927709.1     9444468    9647558   
3                         ArgIR_g10037.t2|KAF8784340.1     9444468    9660874   
4                       ArgIR_g10040.t2|XP_055927297.1     9857689    9973813   
..                                                 ...         ...        ...   
106                        ArgIR_g9566.t1|KAF8788018.1   118335673  118336929   
107  TRINITY_DN12670_c0_g1_i3_S414_S3.p1|XP_0559293...    80213733   80266326   
108  XP_055940122.1|glutamate_receptor_ionotropic|k...    79030669   79077298   
109             XP_055941311.1|replaced_ArgIR_g9783.t1   130240400  130306005   
110  XP_055950590.1|glutamate_receptor_ionotropic|N...    27262432   27383496   

         chr_id  
0    OX38

## To get inputfile2 do:

### to install samtools

To install this package run one of the following:

`conda install bioconda::samtools`

`conda install bioconda/label/cf201901::samtools`



```sh
samtools faidx genome.fna 
cut -f1,2 genome.fna.fai > chromosome_lengths.txt

# Example

samtools faidx /proj/naiss2023-23-109/Spiders_Project/Data/Genome/Genome_assembly_ASM1534279v1/ncbi_dataset/data/GCA_015342795.1/GCA_015342795.1_ASM1534279v1_genomic.fna 

cut -f1,2 /proj/naiss2023-23-109/Spiders_Project/Data/Genome/Genome_assembly_ASM1534279v1/ncbi_dataset/data/GCA_015342795.1/GCA_015342795.1_ASM1534279v1_genomic.fna.fai > chromosome_lengths.txt

more chromosome_lengths.txt
```

| chr_id              | chr_length |
|---------------------|------------|
| OX387439.1          | 151336912  |
| OX387440.1          | 146539707  |
| OX387441.1          | 146223468  |
| OX387442.1          | 144108507  |
| OX387443.1          | 139820183  |
| OX387445.1          | 139055214  |
| OX387447.1          | 136447494  |
| OX387448.1          | 133229410  |
| OX387449.1          | 130975627  |
| OX387450.1          | 127865338  |
| OX387451.1          | 105101263  |
| OX387446.1          | 137459008  |
| OX387444.1          | 139068055  |
| CANOBD010000001.1   | 14049      |
| CANOBD010000002.1   | 105233     |
| CANOBD010000003.1   | 23352      |
| CANOBD010000004.1   | 144037     |
| CANOBD010000005.1   | 27000      |
| CANOBD010000006.1   | 165968     |
| CANOBD010000007.1   | 101000     |
| CANOBD010000008.1   | 17234      |
| CANOBD010000009.1   | 144406     |
| CANOBD010000010.1   | 68185      |
| CANOBD010000011.1   | 132892     |
| CANOBD010000012.1   | 26458      |
| CANOBD010000013.1   | 69255      |
| CANOBD010000014.1   | 34000      |
| CANOBD010000015.1   | 28128      |
| CANOBD010000016.1   | 52772      |
| OX387452.1          | 14061      |
